# House Value Estimation (RandomForest Inference)

This notebook loads the trained RandomForest pipeline saved by `houses_sklearn_regression.ipynb` and shows how to:
- Inspect model metadata
- Predict on rows from `data/houses.csv`
- Predict from custom inputs (Python dict)



In [1]:
# Imports & artifact load
from pathlib import Path
from joblib import load
import pandas as pd

MODEL_PATH = Path("models/house_value_rf.joblib")
DATA_PATH = Path("data/houses.csv")

assert MODEL_PATH.exists(), f"Missing model: {MODEL_PATH}. Re-run training notebook to export."
artifact = load(MODEL_PATH)
model = artifact["model"]
meta = {k: v for k, v in artifact.items() if k != "model"}
meta


{'metrics': {'model': 'RandomForest',
  'rmse_train': 18090.466511833645,
  'rmse_valid': 49465.72379038693,
  'r2_train': 0.9755183787609992,
  'r2_valid': 0.8132752544635701},
 'sklearn_version': '1.7.2',
 'created_by': 'houses_sklearn_regression.ipynb',
 'features': ['median_income',
  'housing_median_age',
  'total_rooms',
  'total_bedrooms',
  'population',
  'households',
  'latitude',
  'longitude'],
 'target': 'median_house_value'}

In [2]:
# Load data and align columns
assert DATA_PATH.exists(), f"Missing data: {DATA_PATH}. Run ./setup.sh"
raw = pd.read_csv(DATA_PATH)
features = meta["features"]
target = meta.get("target", "median_house_value")

X = raw[features].copy()
y = raw[target].copy() if target in raw.columns else None

X.head()


,median_income,housing_median_age,total_rooms,total_bedrooms,population,households,latitude,longitude
0,8.3252,41.0,880.0,129.0,322.0,126.0,37.88,-122.23
1,8.3014,21.0,7099.0,1106.0,2401.0,1138.0,37.86,-122.22
2,7.2574,52.0,1467.0,190.0,496.0,177.0,37.85,-122.24
3,5.6431,52.0,1274.0,235.0,558.0,219.0,37.85,-122.25
4,3.8462,52.0,1627.0,280.0,565.0,259.0,37.85,-122.25


In [3]:
# Predict on a sample of rows
y_pred = model.predict(X.head(10))

pred_df = X.head(10).copy()
pred_df[target + "_pred"] = y_pred
if y is not None:
    pred_df[target] = y.head(10).values
pred_df


,median_income,housing_median_age,total_rooms,total_bedrooms,population,households,latitude,longitude,median_house_value_pred,median_house_value
0,8.3252,41.0,880.0,129.0,322.0,126.0,37.88,-122.23,427755.8050,452600.0
1,8.3014,21.0,7099.0,1106.0,2401.0,1138.0,37.86,-122.22,383565.0300,358500.0
2,7.2574,52.0,1467.0,190.0,496.0,177.0,37.85,-122.24,375583.0175,352100.0
3,5.6431,52.0,1274.0,235.0,558.0,219.0,37.85,-122.25,341915.0450,341300.0
4,3.8462,52.0,1627.0,280.0,565.0,259.0,37.85,-122.25,295105.5000,342200.0
5,4.0368,52.0,919.0,213.0,413.0,193.0,37.85,-122.25,246727.2500,269700.0
6,3.6591,52.0,2535.0,489.0,1094.0,514.0,37.84,-122.25,252989.7525,299200.0
7,3.1200,52.0,3104.0,687.0,1157.0,647.0,37.84,-122.25,244246.5000,241400.0
8,2.0804,42.0,2555.0,665.0,1206.0,595.0,37.84,-122.26,197000.7500,226700.0
9,3.6912,52.0,3549.0,707.0,1551.0,714.0,37.84,-122.25,257772.2500,261100.0


In [4]:
# Helper: predict from a Python dict (single or list)
import pandas as pd

def predict_from_dict(record_or_records: dict | list[dict]):
    df = pd.DataFrame([record_or_records] if isinstance(record_or_records, dict) else record_or_records)
    # Align to training feature order; missing cols become NaN and will be imputed by pipeline
    for col in features:
        if col not in df.columns:
            df[col] = pd.NA
    df = df[features]
    return model.predict(df)

# Example single prediction
example = {
    "median_income": 5.0,
    "housing_median_age": 30,
    "total_rooms": 1500,
    "total_bedrooms": 300,
    "population": 800,
    "households": 300,
    "latitude": 34.05,
    "longitude": -118.25,
}
predict_from_dict(example)


array([218313.0075])